# **HOMEWORK ASSIGNMENT 7c**



In [ ]:
# Local setup: install packages once from the terminal with `python -m pip install -r requirements.txt`.


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "README.md").exists()
    and (PROJECT_ROOT / "requirements.txt").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "requirements.txt").exists():
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from the DataManagement-BusinessIntelligence folder."
    )



In [3]:
# Set-up
db_path = (PROJECT_ROOT / "NYPD_Material" / "star_schema_nypd_complaints.db").resolve().as_posix()
connection_string = f"sqlite:///{db_path}"

%reload_ext sql
%sql $connection_string


In [4]:
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


name
borough_dim
precinct_dim
station_dim
park_dim
nycha_dev_dim
law_category_dim
offense_key_dim
pd_code_dim
jurisdiction_dim
location_dim


In [5]:
%%sql

SELECT * FROM offense_key_dim LIMIT 2;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


ky_cd,ofns_desc
101,MURDER & NON-NEGL. MANSLAUGHTER
104,RAPE


# **1** : Neighborhood, year, and most common crime per year, number of times it repeated, starting from 2010, across all neighborhoods.

In [6]:
%%sql

select * from borough_dim;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


boro_id,boro_nm
1,MANHATTAN
2,STATEN ISLAND
3,BROOKLYN
4,(null)
5,QUEENS
6,BRONX


In [7]:
%%sql

SELECT * FROM complaint_fact LIMIT 2;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


cmplnt_num,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_to_dt,cmplnt_to_tm,rpt_dt,crm_atpt_cptd_cd,boro_id,law_cat_id,ky_cd,pd_cd,jurisdiction_code,precinct_id,station_id,park_id,nycha_dev_id,location_id,victim_profile_id,suspect_profile_id,year,month,hour
303330076,2024-09-28,12:00:00,2024-09-28,14:00:00,2025-03-20,COMPLETED,1,None,341,339,0,1,1,1,1,1,1,1,2024,9,12
301019512,2024-02-13,04:50:00,2024-02-13,04:55:00,2025-02-13,COMPLETED,2,None,121,269,0,2,1,1,1,2,1,2,2024,2,4


## **Work step-by-step**

We first find the count of crime by year and by crime in the temporary *counts table*

We create a table named *maxes* that contains the most frequent offense per year, from the counts table

We print below the year, offense description, offense key, and count using the counts table, joining with the maxes table and the offense_dim table.

**We obtained the most frequent crime per year, across all neighbours.**

In [8]:
%%sql
-- This will become to counts table
-- For each year, you get how many times each crime happened (across all boros)
SELECT year, ky_cd, COUNT(*) AS cnt
  FROM complaint_fact
  GROUP BY year, ky_cd

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


year,ky_cd,cnt
2015,104,5
2015,109,10
2015,110,3
2015,113,1
2015,117,1
2015,118,1
2015,121,2
2015,233,19
2015,340,1
2015,341,3


In [9]:
%%sql

-- This will become the maxes table
-- The table we return here, contains the maximum count per year
WITH counts AS (
  SELECT year, ky_cd, COUNT(*) AS cnt
  FROM complaint_fact
  GROUP BY year, ky_cd)
  SELECT year, MAX(cnt) AS max_cnt
  FROM counts
  GROUP BY year;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


year,max_cnt
2015,19
2016,18
2017,29
2018,21
2019,22
2020,20
2021,41
2022,84
2023,151
2024,1802


In [10]:
%%sql

-- This will come the top_max_table in the next cell
-- We are merging with a join the maxes and counts table
-- We obtain the offense key and description for the crime that happened most frequently by year
WITH counts AS (
  SELECT year, ky_cd, COUNT(*) AS cnt
  FROM complaint_fact
  GROUP BY year, ky_cd
),
maxes AS (
  SELECT year, MAX(cnt) AS max_cnt
  FROM counts
  GROUP BY year
)
SELECT c.year, o.ofns_desc, c.ky_cd, c.cnt
FROM counts c
JOIN maxes m ON c.year = m.year AND c.cnt = m.max_cnt
JOIN offense_key_dim o ON c.ky_cd = o.ky_cd
ORDER BY c.year;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


year,ofns_desc,ky_cd,cnt
2015,SEX CRIMES,233,19
2016,SEX CRIMES,233,18
2017,SEX CRIMES,233,29
2018,SEX CRIMES,233,21
2019,SEX CRIMES,233,22
2020,GRAND LARCENY,109,20
2021,GRAND LARCENY,109,41
2022,GRAND LARCENY,109,84
2023,GRAND LARCENY,109,151
2024,PETIT LARCENY,341,1802


In [11]:
%%sql

-- This is the answer to question 1.
WITH counts AS (
  SELECT year, ky_cd, COUNT(*) AS cnt
  FROM complaint_fact
  GROUP BY year, ky_cd
),
maxes AS (
  SELECT year, MAX(cnt) AS max_cnt
  FROM counts
  GROUP BY year
),
-- top per year is what we returned before
top_per_year AS (
  SELECT c.year, o.ofns_desc, c.ky_cd, c.cnt
  FROM counts c
  JOIN maxes m ON c.year = m.year AND c.cnt = m.max_cnt
  JOIN offense_key_dim o ON c.ky_cd = o.ky_cd
)
SELECT
  t.year,
  bd.boro_nm,
  t.ky_cd,
  t.ofns_desc,
  COUNT(*) AS n_in_borough
  -- we are counting from this join table
FROM top_per_year t
JOIN complaint_fact cf
  ON cf.year = t.year
 AND cf.ky_cd = t.ky_cd
JOIN borough_dim bd
  ON bd.boro_id = cf.boro_id
WHERE t.year >= 2010
GROUP BY t.year, bd.boro_id
-- we were grouping by year and boro_id in class
-- Milena brought up a great point, we might have multiple crimes corresponding the max number of crimes (if sex crimes happens with the same frequency of larceny, for example). In that case, we would count the contribution for both crimes. Grouping using the crime id (ky_cd), as below removes this issue.
ORDER BY t.year, bd.boro_nm, t.ky_cd;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


year,boro_nm,ky_cd,ofns_desc,n_in_borough
2015,BRONX,233,SEX CRIMES,6
2015,BROOKLYN,233,SEX CRIMES,2
2015,MANHATTAN,233,SEX CRIMES,5
2015,QUEENS,233,SEX CRIMES,6
2016,BRONX,233,SEX CRIMES,4
2016,BROOKLYN,233,SEX CRIMES,5
2016,MANHATTAN,233,SEX CRIMES,6
2016,QUEENS,233,SEX CRIMES,3
2017,BRONX,233,SEX CRIMES,9
2017,BROOKLYN,233,SEX CRIMES,11


# **2**: Neighborhood and number of crimes reported in 2024 by neighborhood.

In [12]:
%%sql
-- There is literally a string '(null)' to identify null values
SELECT *
FROM borough_dim b
WHERE b.boro_nm = '(null)';

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


boro_id,boro_nm
4,(null)


In [13]:
%%sql

-- Now we are excluding crimes corresponding to the unspecified neighbords
SELECT b.boro_nm AS neighborhood, COUNT(*) AS total_crimes_2024
FROM complaint_fact AS c
JOIN borough_dim AS b ON c.boro_id = b.boro_id
WHERE c.year = 2024 AND b.boro_nm != '(null)'
GROUP BY b.boro_nm
ORDER BY b.boro_nm;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


neighborhood,total_crimes_2024
BRONX,1577
BROOKLYN,2061
MANHATTAN,1866
QUEENS,1777
STATEN ISLAND,372


#**3** : Year and number of crimes per year in Manhattan from 2010.

In [14]:
%%sql

SELECT c.year, COUNT(*) AS n_crimes_Manhattan
FROM complaint_fact AS c
JOIN borough_dim AS b
  ON c.boro_id = b.boro_id
WHERE b.boro_nm = 'MANHATTAN'
  AND c.year >= 2010
GROUP BY c.year;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


year,n_crimes_Manhattan
2015,16
2016,14
2017,2
2018,12
2019,12
2020,25
2021,26
2022,55
2023,167
2024,1866


# **4**: Most common crime in Manhattan from 2010.

In [15]:
%%sql

-- First step, we identify the total occurences by year and by crime
SELECT
  b.boro_nm, c.year, o.ofns_desc as crime_type, COUNT(*) as total_occurences
  -- The count is done on the join table that contains complaints with the information related to boro and offense
  FROM complaint_fact AS c
  JOIN borough_dim AS b ON c.boro_id = b.boro_id
  JOIN offense_key_dim AS o ON c.ky_cd = o.ky_cd
  WHERE b.boro_nm = 'MANHATTAN' AND c.year >= 2010
  GROUP BY o.ofns_desc;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


boro_nm,year,crime_type,total_occurences
MANHATTAN,2024,ADMINISTRATIVE CODE,4
MANHATTAN,2024,ARSON,2
MANHATTAN,2024,ASSAULT 3 & RELATED OFFENSES,77
MANHATTAN,2024,BURGLARY,30
MANHATTAN,2024,CRIMINAL MISCHIEF & RELATED OF,116
MANHATTAN,2024,CRIMINAL TRESPASS,3
MANHATTAN,2024,DANGEROUS DRUGS,50
MANHATTAN,2024,DANGEROUS WEAPONS,7
MANHATTAN,2024,FELONY ASSAULT,42
MANHATTAN,2023,FORGERY,24


In [16]:
%%sql

-- Now we identify the max.
-- Another solution could be sorting and taking the highest value, but you will never know if the second and third in order had the same value.
-- To be sure, the code below will report any row that correspond to the maximum count
WITH crime_counts AS(
  SELECT
  b.boro_nm, c.year, o.ofns_desc as crime_type, COUNT(*) as total_occurences
  FROM complaint_fact AS c
  JOIN borough_dim AS b ON c.boro_id = b.boro_id
  JOIN offense_key_dim AS o ON c.ky_cd = o.ky_cd
  WHERE b.boro_nm = 'MANHATTAN' AND c.year >= 2010
  GROUP BY o.ofns_desc)
SELECT crime_type, total_occurences
FROM crime_counts
WHERE total_occurences = (SELECT MAX(total_occurences) FROM crime_counts);

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


crime_type,total_occurences
GRAND LARCENY,644


# **5**: Count of number of victims, divided by age ranges, in Manhattan in 2024.

In [17]:
%%sql

SELECT *
FROM victim_profile_dim
LIMIT 2;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


victim_profile_id,vic_age_group,vic_race,vic_sex
1,45-64,WHITE,F
2,25-44,WHITE,M


In [18]:
%%sql

SELECT DISTINCT(vic_age_group)
FROM victim_profile_dim;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


vic_age_group
45-64
25-44
<18
UNKNOWN
18-24
65+


In [19]:
%%sql

-- This is total number of victims divided by age groups, across boros and years
SELECT v.vic_age_group AS age_group, COUNT(*) AS victim_count
FROM complaint_fact AS c
JOIN borough_dim AS b ON c.boro_id = b.boro_id
JOIN victim_profile_dim AS v ON c.victim_profile_id = v.victim_profile_id
GROUP BY age_group;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


age_group,victim_count
18-24,643
25-44,3291
45-64,1905
65+,867
<18,650
UNKNOWN,1779


In [20]:
%%sql

SELECT v.vic_age_group AS age_group, COUNT(*) AS victim_count
FROM complaint_fact AS c
JOIN borough_dim AS b ON c.boro_id = b.boro_id
JOIN victim_profile_dim AS v ON c.victim_profile_id = v.victim_profile_id
-- consider Manhattan only and 2024
WHERE b.boro_nm = 'MANHATTAN' AND c.year = 2024
GROUP BY age_group;

 * sqlite:////Users/vanessadamario/src/DataManagement-BusinessIntelligence/NYPD_Material/star_schema_nypd_complaints.db
Done.


age_group,victim_count
18-24,134
25-44,633
45-64,361
65+,222
<18,57
UNKNOWN,459
